# Mixed hybrid ablation: `z72 corr-core + z24 HIE-incremental`

Этот notebook строит дополнительный **post-screening ablation candidate** по тому же принципу, что и основные hybrid-наборы:

1. берём correlation core из лучшего corr baseline `z72_a0p5_corr_pruned_top10`;
2. берём HIE-incremental candidates из устойчивого HIE ranking для `z24`;
3. собираем финальный набор через greedy Spearman pruning, чтобы избежать повторов и мультиколлинеарности;
4. прогоняем default screening;
5. сравниваем новый mixed-кандидат с уже сохранёнными результатами screening.

Важно: этот набор является **post-screening ablation**, а не заменой основного train-only feature selection pipeline.

## 0. Imports and project setup

In [1]:
import json
import sys
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

# Change this path if the project lives elsewhere.
PROJECT_ROOT = Path(r"D:\PythonProjects\VKR")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data_processing.functions.klines_dataloader import KlinesDataLoader
from data_processing.functions.stream_indicators import stream_TA_lib
from data_processing.functions.transform_indicators import transform_indicators_df
from data_processing.functions.rolling_z_score_clip import rolling_z_score_clip_df
from backtest.backtest_class import Backtesting

## 1. Configuration

In [2]:
# =====================================================================
# Core protocol
# =====================================================================
SEED = 142
ALL_SYMBOLS = ["BTCUSDT", "DOGEUSDT", "ETHUSDT", "SOLUSDT", "XRPUSDT"]
ACTIONS = [0, 1]

INTERVAL = 240
HORIZON = 10
TRADE_COST = 0.0025
OHLCV_PATH = PROJECT_ROOT / f"data/klines_data/crypto_{INTERVAL}m_bybit_TEST.parquet"

VAL_BARS = 2000
TEST_BARS = 2000
EMBARGO_BARS = 0

STATE_FEATURES = [
    "state_in_position",
    "state_log_bars_in_position",
    "state_unrealized_pnl",
]

META_COLS = ["timestamp", "symbol", "open", "high", "low", "close", "volume"]

# =====================================================================
# Source artifacts from final alpha=0.5 HIE+corr feature selection
# =====================================================================
FEATURE_SELECTION_ROOT = PROJECT_ROOT / "feature_selection" / "hybrid_stable_hie_corr_outputs_alpha0p5_train_only_v2"
FEATURE_SETS_JSON = FEATURE_SELECTION_ROOT / "feature_sets_hybrid_corr_stable_hie_alpha0p5.json"
FEATURE_META_JSON = FEATURE_SELECTION_ROOT / "feature_set_meta_hybrid_corr_stable_hie_alpha0p5.json"
STABLE_HIE_PER_Z_CSV = FEATURE_SELECTION_ROOT / "stable_hie_per_z_MAIN_alpha0p5.csv"

# Previous screening outputs are used only for comparison, not for constructing the mixed set.
PREVIOUS_SCREENING_DIR = FEATURE_SELECTION_ROOT / "algorithm_specific_screening_defaults_ts20"
PREVIOUS_RANKING_CSV = PREVIOUS_SCREENING_DIR / "stage1_set_validation_ranking.csv"
PREVIOUS_SELECTED_CSV = PREVIOUS_SCREENING_DIR / "selected_top1_corr_and_hybrid_by_bandit_after_screening.csv"

# New output directory for this ablation.
OUTPUT_DIR = FEATURE_SELECTION_ROOT / "mixed_z72corr_z24hie_default_screening"
SCREENING_OUTPUT_DIR = OUTPUT_DIR
DIAGNOSTICS_OUTPUT_DIR = OUTPUT_DIR / "diagnostics"
for d in [OUTPUT_DIR, SCREENING_OUTPUT_DIR, DIAGNOSTICS_OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# =====================================================================
# Mixed feature set construction
# =====================================================================
MIXED_SET_NAME = "z72corr5_z24hie5_a0p5_mixed_hybrid_top10"
CORR_SOURCE_SET = "z72_a0p5_corr_pruned_top10"
HIE_SOURCE_Z_WINDOW = 24
CORR_CORE_K = 5
HIE_TARGET_K = 5
FINAL_K = 10
PRUNE_THRESHOLD = 0.85

# Prefixes make it explicit that features come from different rolling-z datasets.
Z72_PREFIX = "z72__"
Z24_PREFIX = "z24__"
MIXED_DATASET_KEY = "mixed_z72corr_z24hie"

# Screening mode:
# - "mixed_only": run only the new mixed candidate and compare against previous screening CSVs. Recommended.
# - "mixed_plus_original_8": rerun old 8 feature sets plus the new mixed candidate in one notebook.
SCREENING_MODE = "mixed_only"

# =====================================================================
# Execution defaults fixed during screening
# =====================================================================
START_CAPITAL = 100.0
POSITION_SIZE = 0.10
MIN_HOLD_BARS = 4
COOLDOWN_BARS = 2
CONFIDENCE_THRESHOLD = 0.005
SCREENING_UPDATE_ON_VALIDATION = True

# =====================================================================
# Bandit defaults fixed during screening
# =====================================================================
DEFAULT_DISCOUNT_FACTOR = 0.998
DEFAULT_WINDOW_SIZE = 500
DEFAULT_LAMBDA_PRIOR = 1.0
DEFAULT_NOISE_STD = 0.03
DEFAULT_UCB_ALPHA = 0.10
DEFAULT_REWARD_CLIP = 0.10

BANDIT_SCREENING_CONFIGS = {
    "discounted_lints": {
        "bandit_type": "discounted_lints",
        "discount_factor": DEFAULT_DISCOUNT_FACTOR,
        "lambda_prior": DEFAULT_LAMBDA_PRIOR,
        "noise_std": DEFAULT_NOISE_STD,
        "reward_clip": DEFAULT_REWARD_CLIP,
        "seed": SEED,
    },
    "discounted_linucb": {
        "bandit_type": "discounted_linucb",
        "discount_factor": DEFAULT_DISCOUNT_FACTOR,
        "lambda_prior": DEFAULT_LAMBDA_PRIOR,
        "ucb_alpha": DEFAULT_UCB_ALPHA,
        "reward_clip": DEFAULT_REWARD_CLIP,
        "seed": SEED,
    },
    "sw_lints": {
        "bandit_type": "sw_lints",
        "window_size": DEFAULT_WINDOW_SIZE,
        "lambda_prior": DEFAULT_LAMBDA_PRIOR,
        "noise_std": DEFAULT_NOISE_STD,
        "reward_clip": DEFAULT_REWARD_CLIP,
        "seed": SEED,
    },
    "sw_linucb": {
        "bandit_type": "sw_linucb",
        "window_size": DEFAULT_WINDOW_SIZE,
        "lambda_prior": DEFAULT_LAMBDA_PRIOR,
        "ucb_alpha": DEFAULT_UCB_ALPHA,
        "reward_clip": DEFAULT_REWARD_CLIP,
        "seed": SEED,
    },
}

TS_BANDITS = ["discounted_lints", "sw_lints"]
UCB_BANDITS = ["discounted_linucb", "sw_linucb"]
TS_SCREENING_SEEDS = list(range(142, 172))  # 30 seeds
UCB_SCREENING_SEEDS = [SEED]

CONFIG_FOR_INDICATORS = {
    "ema_periods": [9, 21, 50, 100, 200],
    "momentum_indicators_periods": [14, 30, 72],
    "return_indicators_periods": [6, 24, 72, 168],
    "volatility_indicators_periods": [24, 72, 168],
    "level_periods": [24, 72, 168],
    "vol_ma_period": [24, 72, 168],
    "range_ma_period": [24, 72, 168],
}

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FEATURE_SELECTION_ROOT:", FEATURE_SELECTION_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("SCREENING_MODE:", SCREENING_MODE)

PROJECT_ROOT: D:\PythonProjects\VKR
FEATURE_SELECTION_ROOT: D:\PythonProjects\VKR\feature_selection\hybrid_stable_hie_corr_outputs_alpha0p5_train_only_v2
OUTPUT_DIR: D:\PythonProjects\VKR\feature_selection\hybrid_stable_hie_corr_outputs_alpha0p5_train_only_v2\mixed_z72corr_z24hie_default_screening
SCREENING_MODE: mixed_only


## 2. Load existing feature-selection artifacts

In [3]:
for p in [FEATURE_SETS_JSON, FEATURE_META_JSON, STABLE_HIE_PER_Z_CSV]:
    if not p.exists():
        raise FileNotFoundError(f"Required artifact not found: {p}")

with open(FEATURE_SETS_JSON, "r", encoding="utf-8") as f:
    ORIGINAL_FEATURE_SETS = json.load(f)
with open(FEATURE_META_JSON, "r", encoding="utf-8") as f:
    ORIGINAL_FEATURE_META_RAW = json.load(f)

stable_hie_per_z = pd.read_csv(STABLE_HIE_PER_Z_CSV)

print("Original feature sets:", len(ORIGINAL_FEATURE_SETS))
print("Stable HIE rows:", stable_hie_per_z.shape)
display(stable_hie_per_z.head())

Original feature sets: 8
Stable HIE rows: (117, 29)


,z_window,feature,stability_count,median_union_rank,mean_union_rank,min_union_rank,median_best_rank,median_mean_rank,median_best_hie_norm,mean_best_hie_norm,...,full_train_best_p_value,full_train_regime_coverage_topk,full_train_union_rank_filled,full_train_best_rank_filled,full_train_best_hie_norm_filled,full_train_regime_coverage_topk_filled,stable_hie_rank_within_z,is_stable_ge2,stability_rate,stability_top_k
0,24,ADX_30_scaled,3,5.0,7.000000,4,3.0,6.0,0.017177,0.020136,...,0.0,1.0,4.0,2.0,0.010843,1.0,1,True,1.000000,14
1,24,dist_to_high_24_sqrt_z,3,7.0,6.000000,3,4.0,8.5,0.015101,0.018883,...,NaN,NaN,10000.0,10000.0,-inf,0.0,2,True,1.000000,14
2,24,price_pos_in_range_72_norm,3,9.0,9.666667,6,7.0,12.5,0.018220,0.019961,...,0.0,1.0,2.0,1.0,0.009919,1.0,3,True,1.000000,14
3,24,ema_spread_100_200_z,2,5.5,5.500000,2,4.0,14.0,0.019205,0.019205,...,0.0,1.0,7.0,5.0,0.006919,1.0,4,True,0.666667,14
4,24,RSI_72_bounded,2,6.5,6.500000,3,4.0,7.5,0.015119,0.015119,...,0.0,2.0,1.0,1.0,0.011086,2.0,5,True,0.666667,14


## 3. Helper functions: feature engineering, splits, and mixed dataset merge

In [4]:
def process_indicators_for_z_window(
    ohlcv_data: pd.DataFrame,
    symbols: list[str],
    meta_cols: list[str],
    indicator_config: dict,
    z_window: int,
    clip_value: float = 5.0,
    shift_by_one: bool = True,
) -> pd.DataFrame:
    processed_parts = []
    for symbol in symbols:
        print(f"Processing {symbol}, z_window={z_window}...")
        df_symbol = (
            ohlcv_data[ohlcv_data["symbol"] == symbol]
            .sort_values("timestamp")
            .reset_index(drop=True)
        )
        indicators_symbol = stream_TA_lib(
            df=df_symbol,
            meta_cols=meta_cols,
            **indicator_config,
        )
        indicators_symbol = indicators_symbol.dropna().sort_values("timestamp").reset_index(drop=True)
        transformed_symbol = transform_indicators_df(
            df=indicators_symbol,
            meta_cols=meta_cols,
        )
        z_score_symbol = rolling_z_score_clip_df(
            df=transformed_symbol,
            meta_cols=meta_cols,
            window=z_window,
            clip_value=clip_value,
            shift_by_one=shift_by_one,
        )
        z_score_symbol = z_score_symbol.dropna().sort_values("timestamp").reset_index(drop=True)
        processed_parts.append(z_score_symbol)

    out = (
        pd.concat(processed_parts, ignore_index=True)
        .sort_values(["symbol", "timestamp"])
        .reset_index(drop=True)
    )
    return out


def split_train_val_test_by_tail_bars(
    df: pd.DataFrame,
    symbols: list[str],
    val_bars: int,
    test_bars: int,
    embargo_bars: int = 0,
):
    train_parts, val_parts, test_parts, rows = [], [], [], []
    for sym in symbols:
        g = df[df["symbol"] == sym].sort_values("timestamp").reset_index(drop=True).copy()
        n = len(g)
        if n <= val_bars + test_bars + max(embargo_bars, 0):
            raise ValueError(f"Too few rows for {sym}: {n}")

        test_start = n - test_bars
        val_start = test_start - val_bars
        train_end = max(val_start - embargo_bars, 0)
        val_end = test_start - embargo_bars if embargo_bars > 0 else test_start

        train = g.iloc[:train_end].copy()
        val = g.iloc[val_start:val_end].copy()
        test = g.iloc[test_start:].copy()

        train_parts.append(train)
        val_parts.append(val)
        test_parts.append(test)
        rows.append({
            "symbol": sym,
            "n_total": n,
            "n_train": len(train),
            "n_val": len(val),
            "n_test": len(test),
            "train_start": train["timestamp"].min(),
            "train_end": train["timestamp"].max(),
            "val_start": val["timestamp"].min(),
            "val_end": val["timestamp"].max(),
            "test_start": test["timestamp"].min(),
            "test_end": test["timestamp"].max(),
        })

    train_df = pd.concat(train_parts, ignore_index=True).sort_values(["symbol", "timestamp"]).reset_index(drop=True)
    val_df = pd.concat(val_parts, ignore_index=True).sort_values(["symbol", "timestamp"]).reset_index(drop=True)
    test_df = pd.concat(test_parts, ignore_index=True).sort_values(["symbol", "timestamp"]).reset_index(drop=True)
    return train_df, val_df, test_df, pd.DataFrame(rows)


def get_feature_cols_for_selection(df: pd.DataFrame, meta_cols: list[str]) -> list[str]:
    return [c for c in df.columns if c not in meta_cols]


def validate_feature_columns_exist(df: pd.DataFrame, features: list[str], name: str):
    missing = [f for f in features if f not in df.columns]
    if missing:
        raise ValueError(f"Missing features in {name}: {missing}")


def prefix_feature_columns(df: pd.DataFrame, meta_cols: list[str], prefix: str) -> pd.DataFrame:
    rename = {c: f"{prefix}{c}" for c in df.columns if c not in meta_cols}
    return df.rename(columns=rename)


def merge_prefixed_z_datasets(
    df_z24: pd.DataFrame,
    df_z72: pd.DataFrame,
    meta_cols: list[str],
    z24_prefix: str = Z24_PREFIX,
    z72_prefix: str = Z72_PREFIX,
) -> pd.DataFrame:
    z24p = prefix_feature_columns(df_z24, meta_cols, z24_prefix)
    z72p = prefix_feature_columns(df_z72, meta_cols, z72_prefix)
    merged = (
        z72p.merge(z24p, on=meta_cols, how="inner")
        .sort_values(["symbol", "timestamp"])
        .reset_index(drop=True)
    )
    return merged


def prefixed(raw_feature: str, z_window: int) -> str:
    if int(z_window) == 24:
        return f"{Z24_PREFIX}{raw_feature}"
    if int(z_window) == 72:
        return f"{Z72_PREFIX}{raw_feature}"
    raise ValueError(f"Unsupported mixed z_window source: {z_window}")

## 4. Load OHLCV data and build z24/z72 datasets

In [5]:
loader = KlinesDataLoader(symbols=ALL_SYMBOLS)

ohlcv_data = loader.load_data(
    download_path=str(OHLCV_PATH),
    analyse_data=True,
    cleaning=True,
)

ohlcv_data = (
    ohlcv_data
    .sort_values(["symbol", "timestamp"])
    .reset_index(drop=True)
)

print("OHLCV shape:", ohlcv_data.shape)
print("Date range:", ohlcv_data["timestamp"].min(), "->", ohlcv_data["timestamp"].max())
display(ohlcv_data["symbol"].value_counts())

# Always build z24 and z72 because the mixed candidate uses both.
# If SCREENING_MODE == "mixed_plus_original_8", z48/z96 are built later as well.
Z_WINDOWS_NEEDED = {24, 72}
if SCREENING_MODE == "mixed_plus_original_8":
    Z_WINDOWS_NEEDED.update([24, 48, 72, 96])
Z_WINDOWS_NEEDED = sorted(Z_WINDOWS_NEEDED)
print("Z_WINDOWS_NEEDED:", Z_WINDOWS_NEEDED)

			Статистика без фильтрации

Количество строк: 51672
Количество столбцов: 8
Количество пропущенных значений: 0

Название столбцов: timestamp, open, high, low, close, volume, turnover, symbol
Название активов: BTCUSDT, DOGEUSDT, ETHUSDT, SOLUSDT, XRPUSDT

Длина временного ряда актива BTCUSDT: 10549
Длина временного ряда актива DOGEUSDT: 10209
Длина временного ряда актива ETHUSDT: 10549
Длина временного ряда актива SOLUSDT: 9903
Длина временного ряда актива XRPUSDT: 10462

Временные рамки ряда каждого актива:
BTCUSDT: 2021-07-05 12:00:00+00:00 - 2026-04-28 12:00:00+00:00
DOGEUSDT: 2021-08-31 04:00:00+00:00 - 2026-04-28 12:00:00+00:00
ETHUSDT: 2021-07-05 12:00:00+00:00 - 2026-04-28 12:00:00+00:00
SOLUSDT: 2021-10-21 04:00:00+00:00 - 2026-04-28 12:00:00+00:00
XRPUSDT: 2021-07-20 00:00:00+00:00 - 2026-04-28 12:00:00+00:00


			Статистика после фильтрации (удаление столбца 'turnover' и выравнивание временных рядов по активам)

Количество строк: 49515
Количество столбцов: 7
Количество пропущ

symbol
BTCUSDT     9903
DOGEUSDT    9903
ETHUSDT     9903
SOLUSDT     9903
XRPUSDT     9903
Name: count, dtype: int64

Z_WINDOWS_NEEDED: [24, 72]


## 5. Build datasets by z-window and mixed prefixed dataset

In [6]:
DATASETS_BY_KEY = {}
SPLIT_SUMMARIES = []
PROCESSED_BY_Z = {}

for z_window in Z_WINDOWS_NEEDED:
    print("=" * 100)
    print(f"Building z_window={z_window}")
    print("=" * 100)
    processed = process_indicators_for_z_window(
        ohlcv_data=ohlcv_data,
        symbols=ALL_SYMBOLS,
        meta_cols=META_COLS,
        indicator_config=CONFIG_FOR_INDICATORS,
        z_window=z_window,
        clip_value=5.0,
        shift_by_one=True,
    )
    PROCESSED_BY_Z[z_window] = processed
    train_z, val_z, test_z, split_summary = split_train_val_test_by_tail_bars(
        df=processed,
        symbols=ALL_SYMBOLS,
        val_bars=VAL_BARS,
        test_bars=TEST_BARS,
        embargo_bars=EMBARGO_BARS,
    )
    split_summary.insert(0, "dataset_key", z_window)
    split_summary.insert(1, "z_window", z_window)
    SPLIT_SUMMARIES.append(split_summary)
    DATASETS_BY_KEY[z_window] = {"train": train_z, "val": val_z, "test": test_z}
    print(f"z{z_window}: train={train_z.shape}, val={val_z.shape}, test={test_z.shape}")

# Build mixed dataset on the intersection of z72 and z24 rows, with prefixed feature columns.
print("=" * 100)
print("Building mixed prefixed dataset")
print("=" * 100)
processed_mixed = merge_prefixed_z_datasets(
    df_z24=PROCESSED_BY_Z[24],
    df_z72=PROCESSED_BY_Z[72],
    meta_cols=META_COLS,
)
train_mixed, val_mixed, test_mixed, split_summary_mixed = split_train_val_test_by_tail_bars(
    df=processed_mixed,
    symbols=ALL_SYMBOLS,
    val_bars=VAL_BARS,
    test_bars=TEST_BARS,
    embargo_bars=EMBARGO_BARS,
)
split_summary_mixed.insert(0, "dataset_key", MIXED_DATASET_KEY)
split_summary_mixed.insert(1, "z_window", "z72corr_z24hie")
SPLIT_SUMMARIES.append(split_summary_mixed)
DATASETS_BY_KEY[MIXED_DATASET_KEY] = {"train": train_mixed, "val": val_mixed, "test": test_mixed}

split_summary_all = pd.concat(SPLIT_SUMMARIES, ignore_index=True)
split_summary_all.to_csv(SCREENING_OUTPUT_DIR / "split_summary_mixed_screening.csv", index=False)
print("mixed:", train_mixed.shape, val_mixed.shape, test_mixed.shape)
display(split_summary_all)

Building z_window=24
Processing BTCUSDT, z_window=24...
Processing DOGEUSDT, z_window=24...
Processing ETHUSDT, z_window=24...
Processing SOLUSDT, z_window=24...
Processing XRPUSDT, z_window=24...
z24: train=(28370, 61), val=(10000, 61), test=(10000, 61)
Building z_window=72
Processing BTCUSDT, z_window=72...
Processing DOGEUSDT, z_window=72...
Processing ETHUSDT, z_window=72...
Processing SOLUSDT, z_window=72...
Processing XRPUSDT, z_window=72...
z72: train=(28130, 61), val=(10000, 61), test=(10000, 61)
Building mixed prefixed dataset
mixed: (28130, 115) (10000, 115) (10000, 115)


,dataset_key,z_window,symbol,n_total,n_train,n_val,n_test,train_start,train_end,val_start,val_end,test_start,test_end
0,24,24,BTCUSDT,9674,5674,2000,2000,2021-11-28 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
1,24,24,DOGEUSDT,9674,5674,2000,2000,2021-11-28 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
2,24,24,ETHUSDT,9674,5674,2000,2000,2021-11-28 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
3,24,24,SOLUSDT,9674,5674,2000,2000,2021-11-28 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
4,24,24,XRPUSDT,9674,5674,2000,2000,2021-11-28 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
5,72,72,BTCUSDT,9626,5626,2000,2000,2021-12-06 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
6,72,72,DOGEUSDT,9626,5626,2000,2000,2021-12-06 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
7,72,72,ETHUSDT,9626,5626,2000,2000,2021-12-06 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
8,72,72,SOLUSDT,9626,5626,2000,2000,2021-12-06 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
9,72,72,XRPUSDT,9626,5626,2000,2000,2021-12-06 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00


## 6. Construct mixed feature set with the full hybrid procedure

In [7]:
def max_abs_spearman_against_selected(df: pd.DataFrame, candidate: str, selected: list[str]) -> tuple[float, str | None]:
    if not selected:
        return np.nan, None
    cols = [candidate, *selected]
    corr = df[cols].corr(method="spearman")[candidate].drop(candidate).abs()
    if corr.empty:
        return np.nan, None
    most = corr.idxmax()
    return float(corr.loc[most]), str(most)


def try_add_feature(
    df_train: pd.DataFrame,
    selected: list[str],
    candidate_col: str,
    raw_feature: str,
    source: str,
    phase: str,
    steps: list[dict],
    threshold: float,
) -> bool:
    if candidate_col in selected:
        steps.append({
            "raw_feature": raw_feature,
            "feature_col": candidate_col,
            "source": source,
            "phase": phase,
            "decision": "skipped",
            "reason": "already_selected",
            "max_abs_corr_with_selected": np.nan,
            "most_correlated_selected_feature": None,
        })
        return False
    if candidate_col not in df_train.columns:
        steps.append({
            "raw_feature": raw_feature,
            "feature_col": candidate_col,
            "source": source,
            "phase": phase,
            "decision": "skipped",
            "reason": "missing_column",
            "max_abs_corr_with_selected": np.nan,
            "most_correlated_selected_feature": None,
        })
        return False

    max_corr, most_corr = max_abs_spearman_against_selected(df_train, candidate_col, selected)
    if selected and max_corr >= threshold:
        steps.append({
            "raw_feature": raw_feature,
            "feature_col": candidate_col,
            "source": source,
            "phase": phase,
            "decision": "rejected",
            "reason": f"spearman_ge_{threshold}",
            "max_abs_corr_with_selected": max_corr,
            "most_correlated_selected_feature": most_corr,
        })
        return False

    selected.append(candidate_col)
    steps.append({
        "raw_feature": raw_feature,
        "feature_col": candidate_col,
        "source": source,
        "phase": phase,
        "decision": "selected",
        "reason": "first_feature" if len(selected) == 1 else "below_threshold",
        "max_abs_corr_with_selected": max_corr,
        "most_correlated_selected_feature": most_corr,
    })
    return True


def build_mixed_hybrid_feature_set(
    feature_sets: dict,
    stable_hie_per_z: pd.DataFrame,
    train_mixed: pd.DataFrame,
    corr_source_set: str,
    hie_source_z: int,
    corr_core_k: int,
    hie_target_k: int,
    final_k: int,
    threshold: float,
) -> tuple[list[str], pd.DataFrame, dict]:
    if corr_source_set not in feature_sets:
        raise KeyError(f"corr_source_set not found: {corr_source_set}")

    corr_pruned_reference_raw = list(feature_sets[corr_source_set])
    corr_core_raw = corr_pruned_reference_raw[:corr_core_k]

    hie_candidates = (
        stable_hie_per_z
        .query("z_window == @hie_source_z and is_stable_ge2 == True")
        .sort_values(["stable_hie_rank_within_z", "median_best_p_value", "median_best_hie_norm"], ascending=[True, True, False])
        ["feature"]
        .tolist()
    )
    if len(hie_candidates) == 0:
        raise ValueError(f"No stable HIE candidates found for z_window={hie_source_z}")

    selected = []
    steps = []

    # Phase 1: corr core from z72.
    for raw in corr_core_raw:
        try_add_feature(
            df_train=train_mixed,
            selected=selected,
            candidate_col=prefixed(raw, 72),
            raw_feature=raw,
            source="corr_z72",
            phase="corr_core",
            steps=steps,
            threshold=threshold,
        )

    # Phase 2: HIE incremental from z24, excluding the full corr-pruned reference by raw name.
    n_hie_selected = 0
    for raw in hie_candidates:
        if raw in corr_pruned_reference_raw:
            steps.append({
                "raw_feature": raw,
                "feature_col": prefixed(raw, hie_source_z),
                "source": "hie_z24",
                "phase": "hie_incremental",
                "decision": "skipped",
                "reason": "raw_feature_in_corr_pruned_reference_z72",
                "max_abs_corr_with_selected": np.nan,
                "most_correlated_selected_feature": None,
            })
            continue
        added = try_add_feature(
            df_train=train_mixed,
            selected=selected,
            candidate_col=prefixed(raw, hie_source_z),
            raw_feature=raw,
            source="hie_z24",
            phase="hie_incremental",
            steps=steps,
            threshold=threshold,
        )
        if added:
            n_hie_selected += 1
        if n_hie_selected >= hie_target_k:
            break

    # Phase 3: corr fallback from the rest of z72 corr-pruned top10.
    for raw in corr_pruned_reference_raw[corr_core_k:]:
        if len(selected) >= final_k:
            break
        try_add_feature(
            df_train=train_mixed,
            selected=selected,
            candidate_col=prefixed(raw, 72),
            raw_feature=raw,
            source="corr_z72",
            phase="corr_fallback",
            steps=steps,
            threshold=threshold,
        )

    # Phase 4: HIE fallback from remaining stable HIE candidates.
    for raw in hie_candidates:
        if len(selected) >= final_k:
            break
        try_add_feature(
            df_train=train_mixed,
            selected=selected,
            candidate_col=prefixed(raw, hie_source_z),
            raw_feature=raw,
            source="hie_z24",
            phase="hie_fallback",
            steps=steps,
            threshold=threshold,
        )

    steps_df = pd.DataFrame(steps)
    selected_steps = steps_df[steps_df["decision"].eq("selected")].copy()

    meta = {
        "set_name": MIXED_SET_NAME,
        "family": "mixed_hybrid_corr72_hie24",
        "strategy": "mixed_hybrid_corr72_hie24_full_procedure",
        "method_group": "mixed",
        "z_window": "z72corr_z24hie",
        "dataset_key": MIXED_DATASET_KEY,
        "alpha_out": 0.5,
        "n_features": len(selected),
        "prune_threshold": threshold,
        "corr_source_set": corr_source_set,
        "hie_source_z_window": hie_source_z,
        "corr_core_k": corr_core_k,
        "hie_target_k": hie_target_k,
        "final_k": final_k,
        "corr_pruned_reference_raw": corr_pruned_reference_raw,
        "corr_core_raw": corr_core_raw,
        "hie_candidates_raw_ranked": hie_candidates,
        "selected_features_prefixed": selected,
        "selected_raw_features": selected_steps["raw_feature"].tolist(),
        "selected_sources": selected_steps["source"].tolist(),
    }
    return selected, steps_df, meta


mixed_features, mixed_selection_steps, mixed_meta = build_mixed_hybrid_feature_set(
    feature_sets=ORIGINAL_FEATURE_SETS,
    stable_hie_per_z=stable_hie_per_z,
    train_mixed=train_mixed,
    corr_source_set=CORR_SOURCE_SET,
    hie_source_z=HIE_SOURCE_Z_WINDOW,
    corr_core_k=CORR_CORE_K,
    hie_target_k=HIE_TARGET_K,
    final_k=FINAL_K,
    threshold=PRUNE_THRESHOLD,
)

if len(mixed_features) != FINAL_K:
    raise ValueError(f"Mixed set has {len(mixed_features)} features, expected {FINAL_K}")

FEATURE_SETS = {MIXED_SET_NAME: mixed_features}
FEATURE_SET_META = {MIXED_SET_NAME: mixed_meta}

if SCREENING_MODE == "mixed_plus_original_8":
    # Add original feature sets as optional full rerun candidates.
    for name, feats in ORIGINAL_FEATURE_SETS.items():
        meta = dict(ORIGINAL_FEATURE_META_RAW.get(name, {}))
        meta.setdefault("set_name", name)
        meta.setdefault("features", feats)
        meta.setdefault("n_features", len(feats))
        meta.setdefault("dataset_key", int(meta.get("z_window", __import__("re").search(r"z(\d+)", name).group(1))))
        if "alpha_out" not in meta:
            meta["alpha_out"] = 0.5 if "a0p5" in name else np.nan
        family = meta.get("family", "")
        if not family:
            family = "hybrid_corr_stable_hie" if "hybrid" in name else "corr_pruned"
        meta["family"] = family
        meta["strategy"] = "hybrid_corr_stable_hie" if "hybrid" in name else "corr_pruned"
        meta["method_group"] = "hybrid" if "hybrid" in name else "corr"
        FEATURE_SETS[name] = feats
        FEATURE_SET_META[name] = meta

screening_feature_sets = sorted(
    FEATURE_SETS.keys(),
    key=lambda name: (str(FEATURE_SET_META[name].get("method_group", "")), name),
)

mixed_selection_steps.to_csv(SCREENING_OUTPUT_DIR / "mixed_feature_set_selection_steps.csv", index=False)
with open(SCREENING_OUTPUT_DIR / "mixed_feature_set.json", "w", encoding="utf-8") as f:
    json.dump({MIXED_SET_NAME: mixed_features}, f, ensure_ascii=False, indent=2)
with open(SCREENING_OUTPUT_DIR / "mixed_feature_set_meta.json", "w", encoding="utf-8") as f:
    json.dump({MIXED_SET_NAME: mixed_meta}, f, ensure_ascii=False, indent=2)

print("Mixed features:")
for i, f in enumerate(mixed_features, 1):
    print(f"{i:02d}. {f}")

display(mixed_selection_steps)

Mixed features:
01. z72__range_sqrt_z
02. z72__vol_ratio_72_log1p_z
03. z72__dist_to_high_168_sqrt_z
04. z72__body_sqrt_z
05. z72__bb_width_20_sqrt_z
06. z24__ADX_30_scaled
07. z24__dist_to_high_24_sqrt_z
08. z24__price_pos_in_range_72_norm
09. z24__ema_spread_100_200_z
10. z24__RSI_72_bounded


,raw_feature,feature_col,source,phase,decision,reason,max_abs_corr_with_selected,most_correlated_selected_feature
0,range_sqrt_z,z72__range_sqrt_z,corr_z72,corr_core,selected,first_feature,NaN,NaN
1,vol_ratio_72_log1p_z,z72__vol_ratio_72_log1p_z,corr_z72,corr_core,selected,below_threshold,0.712369,z72__range_sqrt_z
2,dist_to_high_168_sqrt_z,z72__dist_to_high_168_sqrt_z,corr_z72,corr_core,selected,below_threshold,0.057042,z72__vol_ratio_72_log1p_z
3,body_sqrt_z,z72__body_sqrt_z,corr_z72,corr_core,selected,below_threshold,0.694171,z72__range_sqrt_z
4,bb_width_20_sqrt_z,z72__bb_width_20_sqrt_z,corr_z72,corr_core,selected,below_threshold,0.340375,z72__range_sqrt_z
5,ADX_30_scaled,z24__ADX_30_scaled,hie_z24,hie_incremental,selected,below_threshold,0.213639,z72__dist_to_high_168_sqrt_z
6,dist_to_high_24_sqrt_z,z24__dist_to_high_24_sqrt_z,hie_z24,hie_incremental,selected,below_threshold,0.306340,z72__bb_width_20_sqrt_z
7,price_pos_in_range_72_norm,z24__price_pos_in_range_72_norm,hie_z24,hie_incremental,selected,below_threshold,0.683956,z72__dist_to_high_168_sqrt_z
8,ema_spread_100_200_z,z24__ema_spread_100_200_z,hie_z24,hie_incremental,selected,below_threshold,0.696242,z24__price_pos_in_range_72_norm
9,RSI_72_bounded,z24__RSI_72_bounded,hie_z24,hie_incremental,selected,below_threshold,0.785981,z24__price_pos_in_range_72_norm


## 7. Internal correlation check for mixed set

In [8]:
def internal_spearman_check(df_train: pd.DataFrame, features: list[str], threshold: float) -> tuple[pd.DataFrame, pd.DataFrame]:
    corr = df_train[features].corr(method="spearman")
    pairs = []
    for i, f1 in enumerate(features):
        for f2 in features[i + 1:]:
            rho = corr.loc[f1, f2]
            pairs.append({
                "feature_1": f1,
                "feature_2": f2,
                "spearman_corr": float(rho),
                "abs_spearman_corr": float(abs(rho)),
                "above_threshold": bool(abs(rho) >= threshold),
            })
    pair_df = pd.DataFrame(pairs).sort_values("abs_spearman_corr", ascending=False).reset_index(drop=True)
    summary = pd.DataFrame([{
        "set_name": MIXED_SET_NAME,
        "n_features": len(features),
        "max_abs_spearman_corr": float(pair_df["abs_spearman_corr"].max()),
        "pass_threshold": bool(pair_df["abs_spearman_corr"].max() < threshold),
        "threshold": threshold,
    }])
    return summary, pair_df

mixed_corr_summary, mixed_corr_pairs = internal_spearman_check(train_mixed, mixed_features, PRUNE_THRESHOLD)
mixed_corr_summary.to_csv(SCREENING_OUTPUT_DIR / "mixed_internal_correlation_summary.csv", index=False)
mixed_corr_pairs.to_csv(SCREENING_OUTPUT_DIR / "mixed_internal_correlation_pairs.csv", index=False)
display(mixed_corr_summary)
display(mixed_corr_pairs.head(20))

if not bool(mixed_corr_summary.loc[0, "pass_threshold"]):
    raise ValueError("Mixed feature set failed internal Spearman correlation threshold")

,set_name,n_features,max_abs_spearman_corr,pass_threshold,threshold
0,z72corr5_z24hie5_a0p5_mixed_hybrid_top10,10,0.785981,True,0.85


,feature_1,feature_2,spearman_corr,abs_spearman_corr,above_threshold
0,z24__price_pos_in_range_72_norm,z24__RSI_72_bounded,0.785981,0.785981,False
1,z72__range_sqrt_z,z72__vol_ratio_72_log1p_z,0.712369,0.712369,False
2,z24__price_pos_in_range_72_norm,z24__ema_spread_100_200_z,0.696242,0.696242,False
3,z72__range_sqrt_z,z72__body_sqrt_z,0.694171,0.694171,False
4,z72__dist_to_high_168_sqrt_z,z24__ema_spread_100_200_z,-0.686312,0.686312,False
5,z72__dist_to_high_168_sqrt_z,z24__price_pos_in_range_72_norm,-0.683956,0.683956,False
6,z24__ema_spread_100_200_z,z24__RSI_72_bounded,0.508816,0.508816,False
7,z72__vol_ratio_72_log1p_z,z72__body_sqrt_z,0.445396,0.445396,False
8,z72__dist_to_high_168_sqrt_z,z24__RSI_72_bounded,-0.419161,0.419161,False
9,z72__range_sqrt_z,z72__bb_width_20_sqrt_z,0.340375,0.340375,False


## 8. Backtest metric and diagnostics helpers

In [9]:
def _get_store(backtesting, mode: str, store_name: str):
    if mode == "train":
        return getattr(backtesting, f"{store_name}_train")
    if mode == "val":
        return getattr(backtesting, f"{store_name}_val")
    raise ValueError("mode должен быть 'train' или 'val'")


def extract_trade_stats_from_log(trade_log: list[dict], trade_cost: float) -> pd.DataFrame:
    if not trade_log:
        return pd.DataFrame()
    events = pd.DataFrame(trade_log).copy()
    if events.empty or "event" not in events.columns:
        return pd.DataFrame()
    trades = []
    open_trade = None
    for _, row in events.sort_values("timestamp").iterrows():
        if row["event"] == "BUY":
            open_trade = row
        elif row["event"] == "SELL" and open_trade is not None:
            entry_price = float(open_trade["price"])
            exit_price = float(row["price"])
            pnl_before_cost = np.log(exit_price / entry_price)
            pnl_after_cost = pnl_before_cost + 2.0 * np.log(1.0 - trade_cost)
            trades.append({
                "entry_timestamp": open_trade["timestamp"],
                "exit_timestamp": row["timestamp"],
                "entry_price": entry_price,
                "exit_price": exit_price,
                "pnl_before_cost": pnl_before_cost,
                "pnl_after_cost": pnl_after_cost,
                "holding_time": pd.to_datetime(row["timestamp"]) - pd.to_datetime(open_trade["timestamp"]),
            })
            open_trade = None
    return pd.DataFrame(trades)


def compute_backtest_metrics(backtesting, sym: str, mode: str, trade_cost: float) -> dict:
    balance_store = _get_store(backtesting, mode, "balance")
    actions_store = _get_store(backtesting, mode, "actions")
    raw_actions_store = _get_store(backtesting, mode, "raw_actions")
    decision_log_store = _get_store(backtesting, mode, "decision_log")
    trade_log_store = _get_store(backtesting, mode, "trade_log")
    reward_log_store = _get_store(backtesting, mode, "reward_log")

    balance = np.asarray(balance_store.get(sym, []), dtype=float)
    actions = np.asarray(actions_store.get(sym, []), dtype=float)
    raw_actions = np.asarray(raw_actions_store.get(sym, []), dtype=float)
    decision_log = pd.DataFrame(decision_log_store.get(sym, []))
    trade_log = trade_log_store.get(sym, [])
    reward_log = pd.DataFrame(reward_log_store.get(sym, []))

    if len(balance) == 0:
        return {"symbol": sym, "mode": mode, "n_decisions": 0}

    start_value = float(balance[0])
    end_value = float(balance[-1])
    return_pct = (end_value / start_value - 1.0) * 100.0
    running_max = np.maximum.accumulate(balance)
    drawdown = balance / (running_max + 1e-12) - 1.0
    max_drawdown_pct = float(drawdown.min() * 100.0)

    rows = {
        "symbol": sym,
        "mode": mode,
        "start_value": start_value,
        "end_value": end_value,
        "return_pct": return_pct,
        "max_drawdown_pct": max_drawdown_pct,
        "drawdown_abs": abs(max_drawdown_pct),
        "raw_action_1_ratio": float(np.mean(raw_actions == 1)) if len(raw_actions) else np.nan,
        "executed_action_1_ratio": float(np.mean(actions == 1)) if len(actions) else np.nan,
        "executed_action_1": int(np.sum(actions == 1)) if len(actions) else 0,
        "raw_action_1": int(np.sum(raw_actions == 1)) if len(raw_actions) else 0,
        "n_decisions": int(len(actions)),
    }

    if not decision_log.empty:
        rows.update({
            "threshold_applied_ratio": float(decision_log["threshold_applied"].mean()),
            "constraint_applied_ratio": float(decision_log["constraint_applied"].mean()),
            "abs_edge_mean": float(decision_log["abs_edge"].mean()),
            "abs_edge_p25": float(decision_log["abs_edge"].quantile(0.25)),
            "abs_edge_p50": float(decision_log["abs_edge"].quantile(0.50)),
            "abs_edge_p75": float(decision_log["abs_edge"].quantile(0.75)),
            "abs_edge_p90": float(decision_log["abs_edge"].quantile(0.90)),
            "uncertainty_0_mean": float(decision_log["uncertainty_0"].mean()),
            "uncertainty_1_mean": float(decision_log["uncertainty_1"].mean()),
            "score_0_std": float(decision_log["score_0"].std()),
            "score_1_std": float(decision_log["score_1"].std()),
        })
    else:
        rows.update({
            "threshold_applied_ratio": np.nan,
            "constraint_applied_ratio": np.nan,
            "abs_edge_mean": np.nan,
            "abs_edge_p25": np.nan,
            "abs_edge_p50": np.nan,
            "abs_edge_p75": np.nan,
            "abs_edge_p90": np.nan,
            "uncertainty_0_mean": np.nan,
            "uncertainty_1_mean": np.nan,
            "score_0_std": np.nan,
            "score_1_std": np.nan,
        })

    if not reward_log.empty and "reward" in reward_log.columns:
        rew = reward_log["reward"].astype(float)
        rows.update({
            "n_reward_updates": int(len(reward_log)),
            "mean_reward_update": float(rew.mean()),
            "std_reward_update": float(rew.std()),
            "reward_abs_p50": float(rew.abs().quantile(0.50)),
            "reward_abs_p90": float(rew.abs().quantile(0.90)),
            "reward_abs_p95": float(rew.abs().quantile(0.95)),
            "reward_p05": float(rew.quantile(0.05)),
            "reward_p95": float(rew.quantile(0.95)),
        })
    else:
        rows.update({
            "n_reward_updates": 0,
            "mean_reward_update": np.nan,
            "std_reward_update": np.nan,
            "reward_abs_p50": np.nan,
            "reward_abs_p90": np.nan,
            "reward_abs_p95": np.nan,
            "reward_p05": np.nan,
            "reward_p95": np.nan,
        })

    trades = extract_trade_stats_from_log(trade_log=trade_log, trade_cost=trade_cost)
    if not trades.empty:
        gross_profit = trades.loc[trades["pnl_after_cost"] > 0, "pnl_after_cost"].sum()
        gross_loss = trades.loc[trades["pnl_after_cost"] < 0, "pnl_after_cost"].sum()
        rows.update({
            "trades": int(len(trades)),
            "win_rate": float((trades["pnl_after_cost"] > 0).mean()),
            "mean_trade_pnl": float(trades["pnl_after_cost"].mean()),
            "median_trade_pnl": float(trades["pnl_after_cost"].median()),
            "total_trade_pnl": float(trades["pnl_after_cost"].sum()),
            "profit_factor": float(gross_profit / abs(gross_loss + 1e-12)),
            "avg_holding": trades["holding_time"].mean(),
        })
    else:
        rows.update({
            "trades": 0,
            "win_rate": np.nan,
            "mean_trade_pnl": np.nan,
            "median_trade_pnl": np.nan,
            "total_trade_pnl": 0.0,
            "profit_factor": np.nan,
            "avg_holding": pd.NaT,
        })
    return rows


def summarize_validation_results(raw_results: pd.DataFrame) -> pd.DataFrame:
    val = raw_results[raw_results["mode"] == "val"].copy()
    if val.empty:
        return pd.DataFrame()
    val["profit_factor_clean"] = val["profit_factor"].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    val["is_active"] = val["trades"] > 0
    val["has_long_actions"] = val["executed_action_1"] > 0
    val["is_positive_return"] = val["return_pct"] > 0
    summary = (
        val.groupby(["bandit_name", "set_name", "seed"], dropna=False)
        .agg(
            family=("family", "first"),
            strategy=("strategy", "first"),
            z_window=("z_window", "first"),
            alpha_out=("alpha_out", "first"),
            n_market_features=("n_market_features", "first"),
            n_symbols=("symbol", "nunique"),
            mean_return_pct=("return_pct", "mean"),
            median_return_pct=("return_pct", "median"),
            min_return_pct=("return_pct", "min"),
            max_return_pct=("return_pct", "max"),
            max_drawdown_abs=("drawdown_abs", "max"),
            mean_drawdown_abs=("drawdown_abs", "mean"),
            median_profit_factor=("profit_factor_clean", "median"),
            mean_profit_factor=("profit_factor_clean", "mean"),
            total_trades=("trades", "sum"),
            mean_trades=("trades", "mean"),
            mean_win_rate=("win_rate", "mean"),
            mean_raw_action_1_ratio=("raw_action_1_ratio", "mean"),
            mean_executed_action_1_ratio=("executed_action_1_ratio", "mean"),
            mean_threshold_applied_ratio=("threshold_applied_ratio", "mean"),
            mean_constraint_applied_ratio=("constraint_applied_ratio", "mean"),
            active_symbols=("is_active", "sum"),
            long_action_symbols=("has_long_actions", "sum"),
            positive_return_symbols=("is_positive_return", "sum"),
            feature_list=("feature_list", "first"),
        )
        .reset_index()
    )
    summary["all_symbols_active"] = summary["active_symbols"] == summary["n_symbols"]
    summary["all_symbols_have_long"] = summary["long_action_symbols"] == summary["n_symbols"]
    summary = summary.sort_values(
        [
            "bandit_name", "all_symbols_active", "all_symbols_have_long",
            "median_return_pct", "mean_return_pct", "max_drawdown_abs",
            "median_profit_factor", "total_trades",
        ],
        ascending=[True, False, False, False, False, True, False, False],
    ).reset_index(drop=True)
    summary["rank_within_bandit"] = summary.groupby("bandit_name").cumcount() + 1
    return summary


def collect_reward_and_decision_diagnostics(bt, run_meta: dict):
    reward_rows, decision_rows, bandit_diag_rows = [], [], []
    for phase in ["train", "val"]:
        reward_log = bt.get_reward_log_frame(phase) if hasattr(bt, "get_reward_log_frame") else pd.DataFrame()
        decision_log = bt.get_decision_log_frame(phase) if hasattr(bt, "get_decision_log_frame") else pd.DataFrame()

        if not reward_log.empty:
            group_cols = ["symbol", "raw_action", "decision_regime"]
            for keys, g in reward_log.groupby(group_cols, dropna=False):
                reward = g["reward"].astype(float)
                reward_rows.append({
                    **run_meta,
                    "phase": phase,
                    "symbol": keys[0],
                    "raw_action": keys[1],
                    "decision_regime": keys[2],
                    "n": len(g),
                    "reward_mean": float(reward.mean()),
                    "reward_std": float(reward.std()),
                    "reward_abs_p50": float(reward.abs().quantile(0.50)),
                    "reward_abs_p75": float(reward.abs().quantile(0.75)),
                    "reward_abs_p90": float(reward.abs().quantile(0.90)),
                    "reward_abs_p95": float(reward.abs().quantile(0.95)),
                    "reward_p05": float(reward.quantile(0.05)),
                    "reward_p95": float(reward.quantile(0.95)),
                    "positive_reward_ratio": float((reward > 0).mean()),
                    "future_log_ret_std": float(g["future_log_ret"].astype(float).std()),
                    "cost_applied_mean": float(g["cost_applied"].astype(float).mean()) if "cost_applied" in g else np.nan,
                })

        if not decision_log.empty:
            for sym, g in decision_log.groupby("symbol", dropna=False):
                decision_rows.append({
                    **run_meta,
                    "phase": phase,
                    "symbol": sym,
                    "n": len(g),
                    "abs_edge_mean": float(g["abs_edge"].mean()),
                    "abs_edge_p10": float(g["abs_edge"].quantile(0.10)),
                    "abs_edge_p25": float(g["abs_edge"].quantile(0.25)),
                    "abs_edge_p50": float(g["abs_edge"].quantile(0.50)),
                    "abs_edge_p75": float(g["abs_edge"].quantile(0.75)),
                    "abs_edge_p90": float(g["abs_edge"].quantile(0.90)),
                    "threshold_applied_ratio": float(g["threshold_applied"].mean()),
                    "constraint_applied_ratio": float(g["constraint_applied"].mean()),
                    "raw_action_1_ratio": float((g["raw_action"] == 1).mean()),
                    "executed_action_1_ratio": float((g["executed_action"] == 1).mean()),
                    "uncertainty_0_mean": float(g["uncertainty_0"].mean()),
                    "uncertainty_1_mean": float(g["uncertainty_1"].mean()),
                    "score_0_std": float(g["score_0"].std()),
                    "score_1_std": float(g["score_1"].std()),
                    "mean_0_std": float(g["mean_0"].std()),
                    "mean_1_std": float(g["mean_1"].std()),
                })

    if hasattr(bt, "get_bandit_diagnostics_frame"):
        diag = bt.get_bandit_diagnostics_frame()
        if not diag.empty:
            for row in diag.to_dict("records"):
                bandit_diag_rows.append({**run_meta, **row})

    return reward_rows, decision_rows, bandit_diag_rows

## 9. Screening runner adapted to dataset keys

In [10]:
def run_screening_for_configs(
    bandit_configs: dict,
    feature_set_names: list[str],
    seeds_by_bandit: dict[str, list[int]],
    output_label: str,
):
    rows = []
    errors = []
    reward_diag_rows = []
    decision_diag_rows = []
    bandit_diag_rows = []

    total_runs = sum(len(seeds_by_bandit[b]) * len(feature_set_names) for b in bandit_configs)
    run_idx = 0

    for bandit_name, bandit_base_config in bandit_configs.items():
        for set_name in feature_set_names:
            meta = FEATURE_SET_META[set_name]
            dataset_key = meta.get("dataset_key", meta.get("z_window"))
            alpha_out_for_backtest = float(meta["alpha_out"])
            selected_features = FEATURE_SETS[set_name]
            train_z = DATASETS_BY_KEY[dataset_key]["train"]
            val_z = DATASETS_BY_KEY[dataset_key]["val"]

            missing_train = [f for f in selected_features if f not in train_z.columns]
            missing_val = [f for f in selected_features if f not in val_z.columns]
            if missing_train or missing_val:
                errors.append({
                    "bandit_name": bandit_name,
                    "set_name": set_name,
                    "error": "missing_features",
                    "missing_train": "|".join(missing_train),
                    "missing_val": "|".join(missing_val),
                })
                continue

            for seed in seeds_by_bandit[bandit_name]:
                run_idx += 1
                print("=" * 120)
                print(f"[{run_idx}/{total_runs}] {output_label} | bandit={bandit_name} | seed={seed} | set={set_name}")
                print("=" * 120)

                config_for_bandit = dict(bandit_base_config)
                config_for_bandit["n_features"] = len(selected_features) + len(STATE_FEATURES)
                config_for_bandit["actions"] = ACTIONS
                config_for_bandit["seed"] = int(seed)

                run_meta = {
                    "run_label": output_label,
                    "bandit_name": bandit_name,
                    "set_name": set_name,
                    "seed": int(seed),
                    "family": meta["family"],
                    "strategy": meta["strategy"],
                    "z_window": str(meta["z_window"]),
                    "dataset_key": str(dataset_key),
                    "alpha_out": alpha_out_for_backtest,
                    "n_market_features": len(selected_features),
                    "n_state_features": len(STATE_FEATURES),
                    "n_total_features": len(selected_features) + len(STATE_FEATURES),
                    "feature_list": "|".join(selected_features),
                    "update_on_validation": SCREENING_UPDATE_ON_VALIDATION,
                    "confidence_threshold": CONFIDENCE_THRESHOLD,
                    "min_hold_bars": MIN_HOLD_BARS,
                    "cooldown_bars": COOLDOWN_BARS,
                    "position_size": POSITION_SIZE,
                }

                try:
                    bt = Backtesting(
                        meta_cols=META_COLS,
                        feature_columns=selected_features,
                        config_for_bandit=config_for_bandit,
                        trade_cost=TRADE_COST,
                        seed=int(seed),
                        update_on_validation=SCREENING_UPDATE_ON_VALIDATION,
                        horizon=HORIZON,
                        min_hold_bars=MIN_HOLD_BARS,
                        cooldown_bars=COOLDOWN_BARS,
                        confidence_threshold=CONFIDENCE_THRESHOLD,
                        alpha_out=alpha_out_for_backtest,
                        state_feature_columns=STATE_FEATURES,
                    )
                    bt.backtest(
                        dataframe_train=train_z,
                        dataframe_val=val_z,
                        symbols=ALL_SYMBOLS,
                        start_capital=START_CAPITAL,
                        position_size=POSITION_SIZE,
                    )

                    for sym in ALL_SYMBOLS:
                        for mode in ["train", "val"]:
                            metrics = compute_backtest_metrics(bt, sym, mode, TRADE_COST)
                            metrics.update(run_meta)
                            for k, v in bandit_base_config.items():
                                metrics[f"bandit_param_{k}"] = v
                            rows.append(metrics)

                    rdiag, ddiag, bdiag = collect_reward_and_decision_diagnostics(bt, run_meta)
                    reward_diag_rows.extend(rdiag)
                    decision_diag_rows.extend(ddiag)
                    bandit_diag_rows.extend(bdiag)

                except Exception as e:
                    err = {**run_meta, "error": repr(e)}
                    errors.append(err)
                    print("ERROR:", err)

    return {
        "results": pd.DataFrame(rows),
        "errors": pd.DataFrame(errors),
        "reward_diagnostics": pd.DataFrame(reward_diag_rows),
        "decision_diagnostics": pd.DataFrame(decision_diag_rows),
        "bandit_diagnostics": pd.DataFrame(bandit_diag_rows),
    }

## 10. Run default screening for mixed candidate

In [11]:
stage1_seeds_by_bandit = {
    bandit_name: (TS_SCREENING_SEEDS if bandit_name in TS_BANDITS else UCB_SCREENING_SEEDS)
    for bandit_name in BANDIT_SCREENING_CONFIGS
}

expected_screening_backtests = sum(
    len(stage1_seeds_by_bandit[b]) * len(screening_feature_sets)
    for b in BANDIT_SCREENING_CONFIGS
)
print("screening_feature_sets:", screening_feature_sets)
print("Expected screening backtests:", expected_screening_backtests)
print(json.dumps(stage1_seeds_by_bandit, ensure_ascii=False, indent=2))

stage1 = run_screening_for_configs(
    bandit_configs=BANDIT_SCREENING_CONFIGS,
    feature_set_names=screening_feature_sets,
    seeds_by_bandit=stage1_seeds_by_bandit,
    output_label="mixed_z72corr_z24hie_default_screening",
)

stage1_results = stage1["results"]
stage1_errors = stage1["errors"]
stage1_reward_diagnostics = stage1["reward_diagnostics"]
stage1_decision_diagnostics = stage1["decision_diagnostics"]
stage1_bandit_diagnostics = stage1["bandit_diagnostics"]

stage1_results.to_csv(SCREENING_OUTPUT_DIR / "stage1_screening_results_raw.csv", index=False)
stage1_errors.to_csv(SCREENING_OUTPUT_DIR / "stage1_screening_errors.csv", index=False)
stage1_reward_diagnostics.to_csv(DIAGNOSTICS_OUTPUT_DIR / "stage1_backtest_reward_diagnostics.csv", index=False)
stage1_decision_diagnostics.to_csv(DIAGNOSTICS_OUTPUT_DIR / "stage1_decision_edge_diagnostics.csv", index=False)
stage1_bandit_diagnostics.to_csv(DIAGNOSTICS_OUTPUT_DIR / "stage1_bandit_diagnostics.csv", index=False)

print("results:", stage1_results.shape)
print("errors:", stage1_errors.shape)
display(stage1_errors.head())

screening_feature_sets: ['z72corr5_z24hie5_a0p5_mixed_hybrid_top10']
Expected screening backtests: 62
{
  "discounted_lints": [
    142,
    143,
    144,
    145,
    146,
    147,
    148,
    149,
    150,
    151,
    152,
    153,
    154,
    155,
    156,
    157,
    158,
    159,
    160,
    161,
    162,
    163,
    164,
    165,
    166,
    167,
    168,
    169,
    170,
    171
  ],
  "discounted_linucb": [
    142
  ],
  "sw_lints": [
    142,
    143,
    144,
    145,
    146,
    147,
    148,
    149,
    150,
    151,
    152,
    153,
    154,
    155,
    156,
    157,
    158,
    159,
    160,
    161,
    162,
    163,
    164,
    165,
    166,
    167,
    168,
    169,
    170,
    171
  ],
  "sw_linucb": [
    142
  ]
}
[1/62] mixed_z72corr_z24hie_default_screening | bandit=discounted_lints | seed=142 | set=z72corr5_z24hie5_a0p5_mixed_hybrid_top10
BTCUSDT: фаза train началась: 2021-12-06 08:00:00+00:00
BTCUSDT: фаза train закончилась: 2024-06-30 20:00:00+

""


## 11. Aggregate mixed screening results

In [12]:
stage1_seed_level_summary = summarize_validation_results(stage1_results)
stage1_seed_level_summary.to_csv(SCREENING_OUTPUT_DIR / "stage1_seed_level_validation_summary.csv", index=False)
display(stage1_seed_level_summary)


def aggregate_seed_level_ranking(seed_level_summary: pd.DataFrame, output_rank_col: str = "rank_within_bandit") -> pd.DataFrame:
    if seed_level_summary.empty:
        return pd.DataFrame()

    out = (
        seed_level_summary
        .groupby(["bandit_name", "set_name"], dropna=False)
        .agg(
            family=("family", "first"),
            strategy=("strategy", "first"),
            z_window=("z_window", "first"),
            alpha_out=("alpha_out", "first"),
            n_market_features=("n_market_features", "first"),
            n_seeds=("seed", "nunique"),
            seeds_used=("seed", lambda x: "|".join(str(int(v)) for v in sorted(set(x)))),

            median_of_seed_median_return_pct=("median_return_pct", "median"),
            mean_of_seed_median_return_pct=("median_return_pct", "mean"),
            std_of_seed_median_return_pct=("median_return_pct", "std"),
            min_seed_median_return_pct=("median_return_pct", "min"),
            max_seed_median_return_pct=("median_return_pct", "max"),

            median_of_seed_mean_return_pct=("mean_return_pct", "median"),
            median_max_drawdown_abs=("max_drawdown_abs", "median"),
            median_profit_factor=("median_profit_factor", "median"),
            median_total_trades=("total_trades", "median"),
            median_positive_return_symbols=("positive_return_symbols", "median"),
            median_active_symbols=("active_symbols", "median"),
            median_long_action_symbols=("long_action_symbols", "median"),

            all_symbols_active_share=("all_symbols_active", "mean"),
            all_symbols_have_long_share=("all_symbols_have_long", "mean"),
            feature_list=("feature_list", "first"),
        )
        .reset_index()
    )

    out["std_of_seed_median_return_pct"] = out["std_of_seed_median_return_pct"].fillna(0.0)
    out["method_group"] = out["set_name"].map(lambda s: FEATURE_SET_META[s]["method_group"])

    out = (
        out
        .sort_values(
            [
                "bandit_name",
                "all_symbols_active_share",
                "all_symbols_have_long_share",
                "median_of_seed_median_return_pct",
                "mean_of_seed_median_return_pct",
                "median_max_drawdown_abs",
                "median_profit_factor",
                "median_total_trades",
            ],
            ascending=[True, False, False, False, False, True, False, False],
        )
        .reset_index(drop=True)
    )
    out[output_rank_col] = out.groupby("bandit_name").cumcount() + 1
    return out


mixed_ranking = aggregate_seed_level_ranking(stage1_seed_level_summary, output_rank_col="rank_within_bandit_current_run")
mixed_ranking.to_csv(SCREENING_OUTPUT_DIR / "mixed_stage1_set_validation_ranking.csv", index=False)
display(mixed_ranking)

,bandit_name,set_name,seed,family,strategy,z_window,alpha_out,n_market_features,n_symbols,mean_return_pct,...,mean_executed_action_1_ratio,mean_threshold_applied_ratio,mean_constraint_applied_ratio,active_symbols,long_action_symbols,positive_return_symbols,feature_list,all_symbols_active,all_symbols_have_long,rank_within_bandit
0,discounted_lints,z72corr5_z24hie5_a0p5_mixed_hybrid_top10,157,mixed_hybrid_corr72_hie24,mixed_hybrid_corr72_hie24_full_procedure,z72corr_z24hie,0.5,10,5,2.420265,...,0.4914,0.0852,0.0736,5,5,3,z72__range_sqrt_z|z72__vol_ratio_72_log1p_z|z7...,True,True,1
1,discounted_lints,z72corr5_z24hie5_a0p5_mixed_hybrid_top10,144,mixed_hybrid_corr72_hie24,mixed_hybrid_corr72_hie24_full_procedure,z72corr_z24hie,0.5,10,5,1.265446,...,0.5045,0.0869,0.0815,5,5,3,z72__range_sqrt_z|z72__vol_ratio_72_log1p_z|z7...,True,True,2
2,discounted_lints,z72corr5_z24hie5_a0p5_mixed_hybrid_top10,163,mixed_hybrid_corr72_hie24,mixed_hybrid_corr72_hie24_full_procedure,z72corr_z24hie,0.5,10,5,1.567383,...,0.4882,0.0784,0.0923,5,5,3,z72__range_sqrt_z|z72__vol_ratio_72_log1p_z|z7...,True,True,3
3,discounted_lints,z72corr5_z24hie5_a0p5_mixed_hybrid_top10,152,mixed_hybrid_corr72_hie24,mixed_hybrid_corr72_hie24_full_procedure,z72corr_z24hie,0.5,10,5,1.495598,...,0.5049,0.0854,0.0618,5,5,2,z72__range_sqrt_z|z72__vol_ratio_72_log1p_z|z7...,True,True,4
4,discounted_lints,z72corr5_z24hie5_a0p5_mixed_hybrid_top10,148,mixed_hybrid_corr72_hie24,mixed_hybrid_corr72_hie24_full_procedure,z72corr_z24hie,0.5,10,5,4.647808,...,0.5007,0.0818,0.0783,5,5,2,z72__range_sqrt_z|z72__vol_ratio_72_log1p_z|z7...,True,True,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57,sw_lints,z72corr5_z24hie5_a0p5_mixed_hybrid_top10,162,mixed_hybrid_corr72_hie24,mixed_hybrid_corr72_hie24_full_procedure,z72corr_z24hie,0.5,10,5,-1.416392,...,0.4805,0.0678,0.1035,5,5,1,z72__range_sqrt_z|z72__vol_ratio_72_log1p_z|z7...,True,True,27
58,sw_lints,z72corr5_z24hie5_a0p5_mixed_hybrid_top10,149,mixed_hybrid_corr72_hie24,mixed_hybrid_corr72_hie24_full_procedure,z72corr_z24hie,0.5,10,5,-2.544213,...,0.4889,0.0591,0.0929,5,5,1,z72__range_sqrt_z|z72__vol_ratio_72_log1p_z|z7...,True,True,28
59,sw_lints,z72corr5_z24hie5_a0p5_mixed_hybrid_top10,170,mixed_hybrid_corr72_hie24,mixed_hybrid_corr72_hie24_full_procedure,z72corr_z24hie,0.5,10,5,-2.160814,...,0.4912,0.0627,0.1047,5,5,1,z72__range_sqrt_z|z72__vol_ratio_72_log1p_z|z7...,True,True,29
60,sw_lints,z72corr5_z24hie5_a0p5_mixed_hybrid_top10,171,mixed_hybrid_corr72_hie24,mixed_hybrid_corr72_hie24_full_procedure,z72corr_z24hie,0.5,10,5,-2.403348,...,0.4813,0.0618,0.0925,5,5,1,z72__range_sqrt_z|z72__vol_ratio_72_log1p_z|z7...,True,True,30


,bandit_name,set_name,family,strategy,z_window,alpha_out,n_market_features,n_seeds,seeds_used,median_of_seed_median_return_pct,...,median_profit_factor,median_total_trades,median_positive_return_symbols,median_active_symbols,median_long_action_symbols,all_symbols_active_share,all_symbols_have_long_share,feature_list,method_group,rank_within_bandit_current_run
0,discounted_lints,z72corr5_z24hie5_a0p5_mixed_hybrid_top10,mixed_hybrid_corr72_hie24,mixed_hybrid_corr72_hie24_full_procedure,z72corr_z24hie,0.5,10,30,142|143|144|145|146|147|148|149|150|151|152|15...,-2.992772,...,0.796115,606.5,2.0,5.0,5.0,1.0,1.0,z72__range_sqrt_z|z72__vol_ratio_72_log1p_z|z7...,mixed,1
1,discounted_linucb,z72corr5_z24hie5_a0p5_mixed_hybrid_top10,mixed_hybrid_corr72_hie24,mixed_hybrid_corr72_hie24_full_procedure,z72corr_z24hie,0.5,10,1,142,0.747749,...,1.007212,423.0,3.0,5.0,5.0,1.0,1.0,z72__range_sqrt_z|z72__vol_ratio_72_log1p_z|z7...,mixed,1
2,sw_lints,z72corr5_z24hie5_a0p5_mixed_hybrid_top10,mixed_hybrid_corr72_hie24,mixed_hybrid_corr72_hie24_full_procedure,z72corr_z24hie,0.5,10,30,142|143|144|145|146|147|148|149|150|151|152|15...,-4.905297,...,0.756872,590.0,1.0,5.0,5.0,1.0,1.0,z72__range_sqrt_z|z72__vol_ratio_72_log1p_z|z7...,mixed,1
3,sw_linucb,z72corr5_z24hie5_a0p5_mixed_hybrid_top10,mixed_hybrid_corr72_hie24,mixed_hybrid_corr72_hie24_full_procedure,z72corr_z24hie,0.5,10,1,142,0.752091,...,1.005300,466.0,3.0,5.0,5.0,1.0,1.0,z72__range_sqrt_z|z72__vol_ratio_72_log1p_z|z7...,mixed,1


## 12. Compare mixed candidate with previous screening results

In [13]:
def rerank_aggregated_candidates(df: pd.DataFrame, rank_col: str = "rank_with_mixed") -> pd.DataFrame:
    out = df.copy()
    out = (
        out
        .sort_values(
            [
                "bandit_name",
                "all_symbols_active_share",
                "all_symbols_have_long_share",
                "median_of_seed_median_return_pct",
                "mean_of_seed_median_return_pct",
                "median_max_drawdown_abs",
                "median_profit_factor",
                "median_total_trades",
            ],
            ascending=[True, False, False, False, False, True, False, False],
        )
        .reset_index(drop=True)
    )
    out[rank_col] = out.groupby("bandit_name").cumcount() + 1
    return out

if PREVIOUS_RANKING_CSV.exists():
    previous_ranking = pd.read_csv(PREVIOUS_RANKING_CSV)
    previous_ranking["source_run"] = "previous_8sets_screening"
    mixed_for_compare = mixed_ranking.copy()
    mixed_for_compare["source_run"] = "mixed_current_run"

    # Align columns.
    common_cols = sorted(set(previous_ranking.columns) & set(mixed_for_compare.columns))
    combined = pd.concat([previous_ranking[common_cols], mixed_for_compare[common_cols]], ignore_index=True)
    combined_reranked = rerank_aggregated_candidates(combined, rank_col="rank_with_mixed")
    combined_reranked.to_csv(SCREENING_OUTPUT_DIR / "comparison_previous_plus_mixed_reranked.csv", index=False)

    display(combined_reranked.sort_values(["bandit_name", "rank_with_mixed"]))

    # Compact comparison against old best corr, old best hybrid and old overall best.
    rows = []
    for bandit_name, part in combined_reranked.groupby("bandit_name"):
        mixed_row = part[part["set_name"].eq(MIXED_SET_NAME)].iloc[0]
        old_part = part[part["source_run"].eq("previous_8sets_screening")].copy()
        old_best = old_part.sort_values("rank_with_mixed").iloc[0]
        old_best_corr = old_part[old_part["method_group"].eq("corr")].sort_values("rank_with_mixed").iloc[0]
        old_best_hybrid = old_part[old_part["method_group"].eq("hybrid")].sort_values("rank_with_mixed").iloc[0]
        rows.append({
            "bandit_name": bandit_name,
            "mixed_rank_with_mixed": int(mixed_row["rank_with_mixed"]),
            "mixed_return_pct": float(mixed_row["median_of_seed_median_return_pct"]),
            "mixed_drawdown_abs": float(mixed_row["median_max_drawdown_abs"]),
            "mixed_profit_factor": float(mixed_row["median_profit_factor"]),
            "mixed_total_trades": float(mixed_row["median_total_trades"]),
            "old_best_set": old_best["set_name"],
            "old_best_method_group": old_best["method_group"],
            "old_best_return_pct": float(old_best["median_of_seed_median_return_pct"]),
            "mixed_minus_old_best_return_pct": float(mixed_row["median_of_seed_median_return_pct"] - old_best["median_of_seed_median_return_pct"]),
            "old_best_corr_set": old_best_corr["set_name"],
            "old_best_corr_return_pct": float(old_best_corr["median_of_seed_median_return_pct"]),
            "mixed_minus_old_best_corr_return_pct": float(mixed_row["median_of_seed_median_return_pct"] - old_best_corr["median_of_seed_median_return_pct"]),
            "old_best_hybrid_set": old_best_hybrid["set_name"],
            "old_best_hybrid_return_pct": float(old_best_hybrid["median_of_seed_median_return_pct"]),
            "mixed_minus_old_best_hybrid_return_pct": float(mixed_row["median_of_seed_median_return_pct"] - old_best_hybrid["median_of_seed_median_return_pct"]),
        })

    mixed_vs_previous = pd.DataFrame(rows)
    mixed_vs_previous.to_csv(SCREENING_OUTPUT_DIR / "mixed_vs_previous_screening_summary.csv", index=False)
    display(mixed_vs_previous)
else:
    print(f"Previous ranking file not found: {PREVIOUS_RANKING_CSV}")
    print("Only current mixed ranking is available.")

,all_symbols_active_share,all_symbols_have_long_share,alpha_out,bandit_name,family,feature_list,max_seed_median_return_pct,mean_of_seed_median_return_pct,median_active_symbols,median_long_action_symbols,...,min_seed_median_return_pct,n_market_features,n_seeds,seeds_used,set_name,source_run,std_of_seed_median_return_pct,strategy,z_window,rank_with_mixed
0,1.0,1.0,0.5,discounted_lints,hybrid_corr_stable_hie,ret_accel_6_24_z|volatility_72_sqrt_z|vol_rati...,2.895664,-0.203603,5.0,5.0,...,-3.011042,10,30,142|143|144|145|146|147|148|149|150|151|152|15...,z24_a0p5_hybrid_corr5_stablehie5_top10,previous_8sets_screening,1.374068,hybrid_corr_stable_hie,24,1
1,1.0,1.0,0.5,discounted_lints,corr_pruned,ret_accel_6_24_z|volatility_72_sqrt_z|vol_rati...,1.214265,-1.995173,5.0,5.0,...,-5.856775,10,30,142|143|144|145|146|147|148|149|150|151|152|15...,z24_a0p5_corr_pruned_top10,previous_8sets_screening,1.558472,corr_pruned,24,2
2,1.0,1.0,0.5,discounted_lints,hybrid_corr_stable_hie,range_sqrt_z|vol_ratio_24_log1p_z|vol_expansio...,3.365352,-2.111199,5.0,5.0,...,-5.417840,10,30,142|143|144|145|146|147|148|149|150|151|152|15...,z96_a0p5_hybrid_corr5_stablehie5_top10,previous_8sets_screening,1.856783,hybrid_corr_stable_hie,96,3
3,1.0,1.0,0.5,discounted_lints,corr_pruned,range_sqrt_z|vol_ratio_72_log1p_z|dist_to_high...,0.872345,-2.884872,5.0,5.0,...,-5.030462,10,30,142|143|144|145|146|147|148|149|150|151|152|15...,z72_a0p5_corr_pruned_top10,previous_8sets_screening,1.408868,corr_pruned,72,4
4,1.0,1.0,0.5,discounted_lints,mixed_hybrid_corr72_hie24,z72__range_sqrt_z|z72__vol_ratio_72_log1p_z|z7...,1.127681,-2.919384,5.0,5.0,...,-6.305830,10,30,142|143|144|145|146|147|148|149|150|151|152|15...,z72corr5_z24hie5_a0p5_mixed_hybrid_top10,mixed_current_run,1.969156,mixed_hybrid_corr72_hie24_full_procedure,z72corr_z24hie,5
5,1.0,1.0,0.5,discounted_lints,corr_pruned,range_sqrt_z|vol_ratio_72_log1p_z|MACD_hist_pc...,0.589371,-3.495559,5.0,5.0,...,-7.419883,10,30,142|143|144|145|146|147|148|149|150|151|152|15...,z48_a0p5_corr_pruned_top10,previous_8sets_screening,2.027655,corr_pruned,48,6
6,1.0,1.0,0.5,discounted_lints,hybrid_corr_stable_hie,range_sqrt_z|vol_ratio_72_log1p_z|dist_to_high...,-0.386852,-3.955409,5.0,5.0,...,-8.638815,10,30,142|143|144|145|146|147|148|149|150|151|152|15...,z72_a0p5_hybrid_corr5_stablehie5_top10,previous_8sets_screening,1.961369,hybrid_corr_stable_hie,72,7
7,1.0,1.0,0.5,discounted_lints,corr_pruned,range_sqrt_z|vol_ratio_24_log1p_z|vol_expansio...,-1.203492,-3.796158,5.0,5.0,...,-7.948770,10,30,142|143|144|145|146|147|148|149|150|151|152|15...,z96_a0p5_corr_pruned_top10,previous_8sets_screening,1.334921,corr_pruned,96,8
8,1.0,1.0,0.5,discounted_lints,hybrid_corr_stable_hie,range_sqrt_z|vol_ratio_72_log1p_z|MACD_hist_pc...,-0.605628,-3.542643,5.0,5.0,...,-6.516190,10,30,142|143|144|145|146|147|148|149|150|151|152|15...,z48_a0p5_hybrid_corr5_stablehie5_top10,previous_8sets_screening,1.545868,hybrid_corr_stable_hie,48,9
9,1.0,1.0,0.5,discounted_linucb,hybrid_corr_stable_hie,ret_accel_6_24_z|volatility_72_sqrt_z|vol_rati...,1.702106,1.702106,5.0,5.0,...,1.702106,10,1,142,z24_a0p5_hybrid_corr5_stablehie5_top10,previous_8sets_screening,0.000000,hybrid_corr_stable_hie,24,1


,bandit_name,mixed_rank_with_mixed,mixed_return_pct,mixed_drawdown_abs,mixed_profit_factor,mixed_total_trades,old_best_set,old_best_method_group,old_best_return_pct,mixed_minus_old_best_return_pct,old_best_corr_set,old_best_corr_return_pct,mixed_minus_old_best_corr_return_pct,old_best_hybrid_set,old_best_hybrid_return_pct,mixed_minus_old_best_hybrid_return_pct
0,discounted_lints,5,-2.992772,11.022200,0.796115,606.5,z24_a0p5_hybrid_corr5_stablehie5_top10,hybrid,-0.182691,-2.810080,z24_a0p5_corr_pruned_top10,-2.236487,-0.756285,z24_a0p5_hybrid_corr5_stablehie5_top10,-0.182691,-2.810080
1,discounted_linucb,3,0.747749,11.287856,1.007212,423.0,z24_a0p5_hybrid_corr5_stablehie5_top10,hybrid,1.702106,-0.954357,z72_a0p5_corr_pruned_top10,1.635573,-0.887824,z24_a0p5_hybrid_corr5_stablehie5_top10,1.702106,-0.954357
2,sw_lints,7,-4.905297,11.493030,0.756872,590.0,z72_a0p5_corr_pruned_top10,corr,-1.588993,-3.316305,z72_a0p5_corr_pruned_top10,-1.588993,-3.316305,z24_a0p5_hybrid_corr5_stablehie5_top10,-3.947415,-0.957883
3,sw_linucb,1,0.752091,11.044917,1.005300,466.0,z72_a0p5_corr_pruned_top10,corr,0.607042,0.145049,z72_a0p5_corr_pruned_top10,0.607042,0.145049,z24_a0p5_hybrid_corr5_stablehie5_top10,-0.609845,1.361936


## 13. Auto-summary for report / next-stage decision

In [14]:
summary_lines = []
summary_lines.append("Mixed z72-corr + z24-HIE default screening summary")
summary_lines.append(f"Created at: {datetime.now().isoformat(timespec='seconds')}")
summary_lines.append(f"Screening mode: {SCREENING_MODE}")
summary_lines.append(f"Mixed set name: {MIXED_SET_NAME}")
summary_lines.append(f"Expected backtests: {expected_screening_backtests}")
summary_lines.append(f"Errors: {len(stage1_errors)}")
summary_lines.append("")
summary_lines.append("Mixed feature set:")
for i, f in enumerate(mixed_features, 1):
    summary_lines.append(f"  {i:02d}. {f}")
summary_lines.append("")
summary_lines.append("Internal correlation check:")
summary_lines.append(f"  max_abs_spearman_corr={mixed_corr_summary.loc[0, 'max_abs_spearman_corr']:.6f}")
summary_lines.append(f"  pass_threshold={bool(mixed_corr_summary.loc[0, 'pass_threshold'])}")
summary_lines.append("")
summary_lines.append("Current mixed screening ranking:")
for _, row in mixed_ranking.iterrows():
    summary_lines.append(
        f"  {row['bandit_name']}: return={row['median_of_seed_median_return_pct']:.4f}% | "
        f"drawdown={row['median_max_drawdown_abs']:.4f} | PF={row['median_profit_factor']:.4f} | "
        f"trades={row['median_total_trades']:.0f}"
    )

if 'mixed_vs_previous' in globals():
    summary_lines.append("")
    summary_lines.append("Mixed vs previous old-best candidates:")
    for _, row in mixed_vs_previous.iterrows():
        summary_lines.append(
            f"  {row['bandit_name']}: mixed_rank={row['mixed_rank_with_mixed']} | "
            f"mixed_minus_old_best_return={row['mixed_minus_old_best_return_pct']:.4f}% | "
            f"old_best={row['old_best_set']}"
        )

summary_text = "\n".join(summary_lines)
(SCREENING_OUTPUT_DIR / "mixed_screening_summary_for_report.txt").write_text(summary_text, encoding="utf-8")
print(summary_text)

Mixed z72-corr + z24-HIE default screening summary
Created at: 2026-05-15T20:55:09
Screening mode: mixed_only
Mixed set name: z72corr5_z24hie5_a0p5_mixed_hybrid_top10
Expected backtests: 62
Errors: 0

Mixed feature set:
  01. z72__range_sqrt_z
  02. z72__vol_ratio_72_log1p_z
  03. z72__dist_to_high_168_sqrt_z
  04. z72__body_sqrt_z
  05. z72__bb_width_20_sqrt_z
  06. z24__ADX_30_scaled
  07. z24__dist_to_high_24_sqrt_z
  08. z24__price_pos_in_range_72_norm
  09. z24__ema_spread_100_200_z
  10. z24__RSI_72_bounded

Internal correlation check:
  max_abs_spearman_corr=0.785981
  pass_threshold=True

Current mixed screening ranking:
  discounted_lints: return=-2.9928% | drawdown=11.0222 | PF=0.7961 | trades=606
  discounted_linucb: return=0.7477% | drawdown=11.2879 | PF=1.0072 | trades=423
  sw_lints: return=-4.9053% | drawdown=11.4930 | PF=0.7569 | trades=590
  sw_linucb: return=0.7521% | drawdown=11.0449 | PF=1.0053 | trades=466

Mixed vs previous old-best candidates:
  discounted_lints: